In [3]:
import os
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
DATASET_PATH = r"C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26"

TEST_IMAGES = os.path.join(DATASET_PATH, "test", "images")
TEST_LABELS = os.path.join(DATASET_PATH, "test", "labels")

BEST_MODEL = r"C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\yolov26 with yolov26 dataset\model testing\best.pt"

print("Test images:", TEST_IMAGES)
print("Test labels:", TEST_LABELS)
print("Best model:", BEST_MODEL)

print("Images folder exists:", os.path.exists(TEST_IMAGES))
print("Labels folder exists:", os.path.exists(TEST_LABELS))
print("Model exists:", os.path.exists(BEST_MODEL))

Test images: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26\test\images
Test labels: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26\test\labels
Best model: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\yolov26 with yolov26 dataset\model testing\best.pt
Images folder exists: True
Labels folder exists: True
Model exists: True


In [5]:
model = YOLO(BEST_MODEL)

print("Model loaded successfully.")
print("Model:", BEST_MODEL)

Model loaded successfully.
Model: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\yolov26 with yolov26 dataset\model testing\best.pt


In [6]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))




PyTorch: 2.5.1+cu124
CUDA available: True
GPU: NVIDIA RTX A4000


In [7]:
image_files = sorted(
    glob.glob(os.path.join(TEST_IMAGES, "*.jpg")) +
    glob.glob(os.path.join(TEST_IMAGES, "*.jpeg")) +
    glob.glob(os.path.join(TEST_IMAGES, "*.png"))
)

print("Number of test images:", len(image_files))

Number of test images: 57


In [8]:
DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")

metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    conf=0.001,
    device=0,
    plots=True
)

Ultralytics 8.4.115  Python-3.9.13 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A4000, 16375MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.3 GFLOPs
WARNING val: Slow image access detected (ping: 0.00.0 ms, read: 8.50.7 MB/s, size: 44.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26\test\labels... 57 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 57/57 876.2it/s 0.1s
val: New cache created: C:\Users\End-user\Desktop\Eman Mehmood 1\2nd experiment\Pods Detection.v5i.yolo26\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.8it/s 2.2s0.5ss
                   all         57       1045      0.937      0.933       0.97      0.694
Speed: 2.1ms preprocess, 6.5ms inference, 0.0ms loss, 2.3ms postproc

In [9]:
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

f1 = 2 * (precision * recall) / (precision + recall)

print("===== YOLO26n TEST RESULTS =====")
print(f"Precision      : {precision:.4f}")
print(f"Recall         : {recall:.4f}")
print(f"F1-score       : {f1:.4f}")
print(f"mAP@50         : {map50:.4f}")
print(f"mAP@50:95      : {map50_95:.4f}")

===== YOLO26n TEST RESULTS =====
Precision      : 0.9373
Recall         : 0.9330
F1-score       : 0.9352
mAP@50         : 0.9697
mAP@50:95      : 0.6935


In [10]:
def get_ground_truth_count(image_path):
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    label_path = os.path.join(TEST_LABELS, image_name + ".txt")

    if not os.path.exists(label_path):
        return None

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    return len(lines)

In [11]:
results_table = []

for image_path in image_files:

    start_time = time.perf_counter()

    results = model.predict(
        source=image_path,
        conf=0.7,
        imgsz=640,
        device=0,
        verbose=False
    )

    elapsed = time.perf_counter() - start_time

    result = results[0]

    predicted_count = len(result.boxes)
    ground_truth_count = get_ground_truth_count(image_path)

    absolute_error = None

    if ground_truth_count is not None:
        absolute_error = abs(
            ground_truth_count - predicted_count
        )

    results_table.append({
        "image": os.path.basename(image_path),
        "ground_truth_count": ground_truth_count,
        "predicted_count": predicted_count,
        "absolute_error": absolute_error,
        "inference_time_sec": elapsed,
        "inference_time_ms": elapsed * 1000
    })

results_df = pd.DataFrame(results_table)

results_df.head()

,image,ground_truth_count,predicted_count,absolute_error,inference_time_sec,inference_time_ms
0,IMG_0025_jpeg.rf.28ff0e1f3fbf3bac5f89b1eac2f3c...,17,11,6,0.082734,82.7340
1,IMG_0059_jpeg.rf.795a0843311a0044c830e6e571790...,21,15,6,0.034507,34.5070
2,IMG_0062_jpeg.rf.df5c821598de499c270156c06bb08...,19,16,3,0.031924,31.9240
3,IMG_0073_jpeg.rf.ba638fbf170c116a5e13dcdc08b8c...,5,5,0,0.031662,31.6620
4,IMG_0076_jpeg.rf.99bdfe62fe1ccec83e650c3ef3600...,11,9,2,0.031477,31.4767


In [12]:
valid_df = results_df.dropna(
    subset=["ground_truth_count", "predicted_count"]
).copy()

y_true = valid_df["ground_truth_count"].values
y_pred = valid_df["predicted_count"].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

nonzero_mask = y_true != 0

mape = np.mean(
    np.abs(
        (y_true[nonzero_mask] - y_pred[nonzero_mask])
        / y_true[nonzero_mask]
    )
) * 100

print("===== SOYBEAN POD COUNTING RESULTS =====")
print(f"Test images : {len(valid_df)}")
print(f"MAE         : {mae:.4f} pods/image")
print(f"RMSE        : {rmse:.4f} pods/image")
print(f"MAPE        : {mape:.2f}%")
print(f"R²          : {r2:.4f}")

===== SOYBEAN POD COUNTING RESULTS =====
Test images : 57
MAE         : 3.6140 pods/image
RMSE        : 4.8738 pods/image
MAPE        : 18.85%
R²          : 0.7666


In [13]:
OUTPUT_CSV = "YOLO26n_soybean_pod_counting_results.csv"

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Results saved to:", OUTPUT_CSV)

Results saved to: YOLO26n_soybean_pod_counting_results.csv


In [14]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_true,
    y_pred,
    alpha=0.7
)

min_value = min(y_true.min(), y_pred.min())
max_value = max(y_true.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Ground-truth pod count")
plt.ylabel("Predicted pod count")

plt.title(
    f"Soybean Pod Counting: Actual vs Predicted\n"
    f"MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.3f}"
)

plt.grid(True)
plt.tight_layout()

plt.savefig(
    "YOLO26n_actual_vs_predicted_pod_count.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

<Figure size 700x700 with 1 Axes>

In [15]:
PREDICTION_DIR = "YOLO26n_test_predictions"
os.makedirs(PREDICTION_DIR, exist_ok=True)

for image_path in image_files:

    results = model.predict(
        source=image_path,
        conf=0.7,
        imgsz=640,
        device=0,
        verbose=False
    )

    annotated = results[0].plot()

    output_path = os.path.join(
        PREDICTION_DIR,
        os.path.basename(image_path)
    )

    import cv2
    cv2.imwrite(output_path, annotated)

print("Annotated test images saved in:", PREDICTION_DIR)

Annotated test images saved in: YOLO26n_test_predictions


In [16]:
summary = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1-score",
        "mAP@50",
        "mAP@50:95",
        "Counting MAE",
        "Counting RMSE",
        "Counting MAPE",
        "Counting R²"
    ],
    "Value": [
        precision,
        recall,
        f1,
        map50,
        map50_95,
        mae,
        rmse,
        mape,
        r2
    ]
})

summary

,Metric,Value
0,Precision,0.937314
1,Recall,0.933014
2,F1-score,0.935159
3,mAP@50,0.969668
4,mAP@50:95,0.693541
5,Counting MAE,3.614035
6,Counting RMSE,4.873847
7,Counting MAPE,18.854078
8,Counting R²,0.766579
